# Synthetic Time Series Benchmarks & Smoothing Evaluation Demo

This demo notebook showcases the synthetic time series benchmark suite, evaluating a 3-point moving average filter against a naive last-value forecast under controlled noise variance.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0', 'seaborn==0.13.2')

In [ ]:
import json
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# NumPy 2.0 compatibility shims if needed
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-230d6e-robust-temporal-smoothing-evaluating-mov/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data = load_data()
print(f"Loaded {len(data['datasets'])} datasets successfully.")

In [ ]:
# Configuration parameters (tunable)
MAX_DATASETS = 5
WINDOW_SIZE = 3

## Forecast Evaluation: Moving Average vs Naive Forecast

We evaluate a 3-point moving average filter against a naive last-value baseline across the synthetic time series examples.

In [ ]:
results = []

for ds in data["datasets"][:MAX_DATASETS]:
    ds_name = ds["dataset"]
    for ex in ds["examples"]:
        history = json.loads(ex["input"])
        target = float(ex["output"])
        
        # Naive forecast: last value in history
        pred_naive = history[-1]
        
        # Moving average forecast: mean of last WINDOW_SIZE values
        w = min(len(history), WINDOW_SIZE)
        pred_ma = float(np.mean(history[-w:]))
        
        err_naive = abs(pred_naive - target)
        err_ma = abs(pred_ma - target)
        
        results.append({
            "dataset": ds_name,
            "step": ex["metadata_step"],
            "target": target,
            "naive_error": err_naive,
            "ma_error": err_ma
        })

df_results = pd.DataFrame(results)
print(f"Evaluated {len(df_results)} examples across {MAX_DATASETS} datasets.")
display(df_results.head())

## Summary and Visualization

In [ ]:
mean_naive = df_results["naive_error"].mean()
mean_ma = df_results["ma_error"].mean()

print(f"Mean Absolute Error (Naive Forecast): {mean_naive:.4f}")
print(f"Mean Absolute Error (3-point Moving Average): {mean_ma:.4f}")

plt.figure(figsize=(8, 5))
plt.bar(["Naive Forecast", "3-pt Moving Average"], [mean_naive, mean_ma], color=["skyblue", "salmon"])
plt.ylabel("Mean Absolute Error")
plt.title("Forecasting Error Comparison")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()